# Solver API

## `RO_solver`

The `RO_solver` function numerically solves the RO equation using the supplied parameter set and generates ensembles of $T$ and $h$ time series.

## Syntax

In [ ]:
[T, h] = RO_solver(par, IC, N, NE)
[T, h] = RO_solver(par, IC, N, NE, NM)
[T, h] = RO_solver(par, IC, N, NE, NM, dt)
[T, h] = RO_solver(par, IC, N, NE, NM, dt, saveat)
[T, h] = RO_solver(par, IC, N, NE, NM, dt, saveat, savemethod)
[T, h] = RO_solver(par, IC, N, NE, NM, dt, saveat, savemethod, EF)
[T, h] = RO_solver(par, IC, N, NE, NM, dt, saveat, savemethod, EF, noise_custom)
[T, h, noise] = RO_solver(...)

## Description

- `[T,h] = RO_solver(par, IC, N, NE)` returns numerically integrated ensembles of `T` and `h`.
- Adding `NM` specifies the numerical integration method.
- Adding `dt` specifies the numerical time step.
- Adding `saveat` specifies the output interval.
- Adding `savemethod` specifies output post-processing.
- Adding `EF` specifies external forcing `E_T` and `E_h`.
- Adding `noise_custom` specifies a custom stochastic forcing realization.
- Requesting the third output, `noise`, returns the noise time series used in the integration.

## Input arguments

### `par`

Input RO parameter cell array (`16×1`):

In [ ]:
par = {R; F1; F2; epsilon; b_T; c_T; d_T; b_h; ...
       sigma_T; sigma_h; B; m_T; m_h; n_T; n_h; n_g};

For `R`, `F1`, `F2`, `epsilon`, `b_T`, `c_T`, `d_T`, `b_h`, `sigma_T`, `sigma_h`, `B`, `m_T`, and `m_h`, each input may be:

- scalar `X`: seasonally constant parameter
- `[X, Xa, phi_a]`: annual seasonal variation
- `[X, Xa, phi_a, Xsa, phi_sa]`: annual + semiannual seasonal variation
- `N×1` vector: user-specified monthly time-varying parameter; when `dt < 1.0`, linear interpolation fills the internal integration time steps
- `NT×1` vector: user-specified parameter on the numerical integration grid, where `NT` corresponds to the integration time grid

`n_T`, `n_h`, and `n_g` are scalars.

### `IC`

Initial conditions for `T` and `h`, specified as `[T0, h0]` (`2×1` or `1×2`).

### `N`

Simulation length in months (scalar).

### `NE`

Number of ensemble members with different noise realizations (scalar).

### `NM`

Numerical integration method (default = `"EH"`):

- `"EM"` — Euler-Maruyama
- `"EH"` — Euler-Heun

### `dt`

Numerical integration time step in months (scalar; default = `0.1`).

### `saveat`

Output interval in months (scalar; default = `1.0`).

### `savemethod`

Post-processing method for saved output (default = `"sampling"`):

- `"sampling"` — extract raw output every `saveat`
- `"mean"` — apply averaging over the output window specified by `saveat`

### `EF`

External forcing cell array (`2×1`; default = `{0.0, 0.0}`):

```matlab
EF = {E_T; E_h};
```

Each forcing term can be:

- scalar `X`: time-constant forcing
- `[X, Xa, phi_a]`: annual seasonal forcing
- `[X, Xa, phi_a, Xsa, phi_sa]`: annual + semiannual seasonal forcing
- `N×1` monthly time series; linearly interpolated to the integration grid when `dt < 1.0`
- `NT×1` time series specified directly on the numerical integration grid

### `noise_custom`

Custom white noise (default = `[]`).

Accepted forms:

- `[]`: internally generated random noise
- scalar: generate the noise from the specified random seed
- matrix of size `(NT-1)×4`: use the supplied noise time series

For a custom matrix, the four columns correspond to the white-noise terms used in the `dT/dt`, `dh/dt`, `d\xi_T/dt`, and `d\xi_h/dt` equations.

- For a white-noise RO (`n_T = n_h = 1`), only columns 1 and 2 are used.
- For a red-noise RO (`n_T = n_h = 0`), only columns 3 and 4 are used.
- When an explicit noise matrix is supplied, the same noise realization is applied to each ensemble member.

## Output arguments

### `T`

Output ensemble of $T$ time series, returned as an `NT×NE` matrix. With output interval `saveat`, the manual defines the saved output length as approximately `NT = floor(N/saveat)`.

### `h`

Output ensemble of $h$ time series, also returned as an `NT×NE` matrix.

### `noise`

Noise used for numerical integration, returned as an `(NT_internal-1)×4×NE` array, where the internal time grid is determined by `dt`.

## Examples

### Linear-white-additive RO with default solver options

In [ ]:
% Parameters
R       = -0.06;
F1      =  0.02;
F2      =  0.9;
epsilon =  0.03;
b_T     =  0.0;
c_T     =  0.0;
d_T     =  0.0;
b_h     =  0.0;
sigma_T =  0.2;
sigma_h =  1.2;
B       =  0.0;
m_T     =  1.0;
m_h     =  1.0;
n_T     =  1;
n_h     =  1;
n_g     =  0;

par = {R; F1; F2; epsilon; b_T; c_T; d_T; b_h; ...
       sigma_T; sigma_h; B; m_T; m_h; n_T; n_h; n_g};

% Solver options
IC = [1.0; 0.0];   % initial T (K) and h (m)
N  = 12 * 10;      % simulation length (months)
NE = 100;          % number of ensemble members

% Solve
[T, h] = RO_solver(par, IC, N, NE);

### Euler-Maruyama with a finer time step

In [ ]:
NM = "EM";
dt = 0.01;

[T, h] = RO_solver(par, IC, N, NE, NM, dt);